# 01 · KDFU Dataset Exploration

**Thesis:** DFU-LiteNet: A Benchmark-Driven, Explainable and Lightweight Attention-Enhanced CNN for Diabetic Foot Ulcer Classification with Leakage-Free Evaluation

**Goal of this notebook:** understand the dataset before training anything.
1. Count the images in every folder.
2. Check image sizes and colour modes.
3. Look at random samples from each class.

**Dataset:** Kaggle DFU (`laithjj/diabetic-foot-ulcer-dfu`), from Alzubaidi et al., *DFU_QUTNet*, Multimedia Tools and Applications 79 (2020) 15655-15677.

In [ ]:
import random
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from pathlib import Path

# Locate the dataset wherever Kaggle mounted it
PATCHES = next(Path("/kaggle/input").rglob("Patches"))
DATA = PATCHES.parent
CLASSES = {"Abnormal (Ulcer)": PATCHES / "Abnormal(Ulcer)",
           "Normal (Healthy skin)": PATCHES / "Normal(Healthy skin)"}
EXTS = {".jpg", ".jpeg", ".png", ".gif", ".webp"}

def images(folder):
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in EXTS)

print("Dataset folder:", DATA)

## 1. Image counts per folder
Only the two `Patches` folders have class labels, so those are what we train and test on.

In [ ]:
counts = [{"folder": str(f.relative_to(DATA)), "images": len(images(f))}
          for f in sorted(p for p in DATA.rglob("*") if p.is_dir())]
counts = pd.DataFrame(counts).query("images > 0").reset_index(drop=True)
counts

## 2. Image sizes and colour modes (training patches)

In [ ]:
rows = []
for name, folder in CLASSES.items():
    for p in images(folder):
        with Image.open(p) as im:
            rows.append({"class": name, "size": f"{im.size[0]}x{im.size[1]}", "mode": im.mode})
info = pd.DataFrame(rows)

print("Images per class:")
print(info["class"].value_counts(), "\n")
print("Most common sizes per class:")
print(info.groupby("class")["size"].value_counts().groupby(level=0).head(5), "\n")
print("Colour modes:")
print(info.groupby("class")["mode"].value_counts())

**Observation:** almost all patches are 224x224 RGB, and the classes are nearly balanced (512 ulcer vs 543 normal).
The few odd sizes will be resized to 224x224 during preprocessing.

## 3. Random samples from each class

In [ ]:
random.seed(42)
fig, axes = plt.subplots(4, 8, figsize=(16, 8.8))
fig.subplots_adjust(left=0.01, right=0.99, top=0.91, bottom=0.01, wspace=0.06, hspace=0.18)
for block, (name, folder) in enumerate(CLASSES.items()):
    for i, p in enumerate(random.sample(images(folder), 16)):
        ax = axes[i // 4, block * 4 + i % 4]
        ax.imshow(Image.open(p).convert("RGB"))
        ax.set_title(p.name, fontsize=8)
        ax.axis("off")
    fig.text(0.25 + 0.5 * block, 0.96, name, ha="center", fontsize=15, fontweight="bold")
plt.savefig("kdfu_sample_grid.png", dpi=110)
plt.show()

## Summary
- **1,055 labelled patches:** 512 ulcer and 543 normal skin, almost all 224x224 RGB.
- `Original Images` (493 full foot photos) are the photos the patches were cut from. We'll use them later to group patches by source photo.
- `TestSet` has no labels, so it can't be used for scoring.
- In the sample grid, ulcer patches **18.jpg** and **185.jpg** show the same wound, which hints at duplicates. Notebook 02 checks this properly.